In [ ]:
import os, subprocess, sys

# 1. Clone your Movio repository (or upload as a Kaggle dataset)
!git clone https://github.com/tripathiji1312/movio-tts.git movio
%cd movio

# 2. Install dependencies
# NOTE: Do NOT install f5-tts from pip — it shadows the vendored IndicF5 and produces garbage.
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate soundfile librosa jieba ctranslate2 fastapi uvicorn websockets pydantic pyyaml torchdiffeq x_transformers pypinyin vocos redis huggingface_hub pydub safetensors nltk

# 3. Check GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

In [ ]:
# Config is already correct in the repo (device=cuda, steps=8, cfg=1.0).
# Just verify it loaded properly.
import yaml

with open("config/settings.yaml", "r") as f:
    cfg = yaml.safe_load(f)

indicf5 = cfg["stage_c"]["indicf5"]
print(f"device: {indicf5['device']}")
print(f"num_flow_steps: {indicf5['num_flow_steps']}")
print(f"cfg_strength: {indicf5['cfg_strength']}")
print(f"speed: {indicf5['speed']}")

# Verify reference audio files exist
import os
for voice_dir in ["config/voices/ta_female_neutral", "config/voices/ta_male_neutral"]:
    ref = os.path.join(voice_dir, "ref.wav")
    print(f"{ref}: {'OK' if os.path.exists(ref) else 'MISSING'}")

In [3]:
import os

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

from huggingface_hub import whoami
print('authenticated as:', whoami()['name'])

authenticated as: tripathiji1312


In [ ]:
import subprocess, time, re

# 1. Download cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2. Start Movio server in the background
server_proc = subprocess.Popen([sys.executable, "-m", "movio"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Movio server starting on port 8000...")
time.sleep(10)  # give model loading more time on first run

# 3. Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 4. Extract and print public URL
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

print("\n" + "="*60)
print(f"LIVE GPU WEB UI & API: {tunnel_url}")
print(f"WebSocket Stream URL : {tunnel_url.replace('https://', 'wss://')}/tts/stream")
print("="*60 + "\n")

# Keep the cell alive and stream server logs
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="")
except KeyboardInterrupt:
    server_proc.terminate()
    tunnel_proc.terminate()